# Soil Fertility Classification Using Machine Learning
### Project Based Learning (PBL) — BCA

**Objective:** Classify soil into Low Fertility, Medium Fertility and High Fertility using Decision Tree, Random Forest, KNN and SVM.

**Dataset note:** The supplied README describes the 1,000 rows as source-calibrated synthetic teaching data, not 1,000 authentic laboratory observations. `Fertility_Class` is a derived teaching label based on N, P, K, organic carbon, pH and EC. Model results should not be claimed as validated real-world soil diagnosis.

## Cell 1 — Project objective and problem statement
This project uses soil properties and contextual information to predict one of three fertility classes. The workflow includes data understanding, cleaning, missing-value handling, EDA, visualization, correlation, feature selection, model training, evaluation, interpretation and prediction.

## Cell 2 — Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## Cell 3 — Load the CSV dataset

In [ ]:
df = pd.read_csv("soil_fertility_classification_1000.csv")
print("Dataset loaded successfully")
print("Shape:", df.shape)
display(df.head())

## Cell 4 — Dataset dimensions and column names

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

## Cell 5 — Dataset information and data types

In [ ]:
df.info()

## Cell 6 — Statistical summary

In [ ]:
display(df.describe(include="all").T)

## Cell 7 — Understand dataset attributes
The dataset contains sample ID, source information, location, soil type, crop, soil measurements, a derived fertility score and the target fertility class.

In [ ]:
attribute_meaning = pd.DataFrame({
    "Attribute": ["pH", "Electrical_Conductivity_dS_m", "Organic_Carbon_g_kg",
                  "Available_N_kg_ha", "Available_P_kg_ha", "Available_K_kg_ha",
                  "Sulphur_mg_kg", "Zinc_mg_kg", "Boron_mg_kg", "Iron_mg_kg",
                  "Manganese_mg_kg", "Copper_mg_kg", "Fertility_Class"],
    "Meaning": ["Soil acidity/alkalinity", "Electrical conductivity",
                "Organic carbon", "Available nitrogen", "Available phosphorus",
                "Available potassium", "Sulphur", "Zinc", "Boron", "Iron",
                "Manganese", "Copper", "Derived target class"]
})
display(attribute_meaning)

## Cell 8 — Check missing values

In [ ]:
missing = df.isnull().sum().to_frame("Missing_Count")
display(missing)
print("Total missing values:", df.isnull().sum().sum())

## Cell 9 — Check duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Sample_ID values:", df["Sample_ID"].duplicated().sum())

## Cell 10 — Check inconsistent or invalid values

In [ ]:
print("Target labels:", df["Fertility_Class"].unique())
print("\nNegative numerical values:")
display((df.select_dtypes(include=np.number) < 0).sum().to_frame("Negative_Count"))
print("\npH range:", df["pH"].min(), "to", df["pH"].max())

## Cell 11 — Handle missing values
The supplied file has no missing values. For modelling, imputers are still included in the preprocessing pipeline so that future samples with missing values can be handled.

In [ ]:
# Keep a clean copy; no rows need to be removed based on the checks above.
df_clean = df.copy()
print("Clean dataset shape:", df_clean.shape)
print("Missing values after check:", df_clean.isnull().sum().sum())

## Cell 12 — Target class distribution

In [ ]:
class_counts = df_clean["Fertility_Class"].value_counts()
display(class_counts.to_frame("Samples"))
plt.figure(figsize=(7,4))
sns.countplot(data=df_clean, x="Fertility_Class",
              order=["Low Fertility", "Medium Fertility", "High Fertility"])
plt.title("Distribution of Soil Fertility Classes")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Cell 13 — Histograms of numerical soil properties

In [ ]:
soil_numeric_cols = ["pH", "Electrical_Conductivity_dS_m", "Organic_Carbon_g_kg",
    "Available_N_kg_ha", "Available_P_kg_ha", "Available_K_kg_ha",
    "Sulphur_mg_kg", "Zinc_mg_kg", "Boron_mg_kg", "Iron_mg_kg",
    "Manganese_mg_kg", "Copper_mg_kg"]
df_clean[soil_numeric_cols].hist(figsize=(15,12), bins=20, edgecolor="black")
plt.suptitle("Distribution of Numerical Soil Properties")
plt.tight_layout()
plt.show()

## Cell 14 — Boxplots by fertility class

In [ ]:
for col in soil_numeric_cols:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df_clean, x="Fertility_Class", y=col,
                order=["Low Fertility", "Medium Fertility", "High Fertility"])
    plt.title(f"{col} by Fertility Class")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## Cell 15 — Soil type and crop distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.countplot(data=df_clean, y="Soil_Type", order=df_clean["Soil_Type"].value_counts().index, ax=axes[0])
axes[0].set_title("Soil Type Distribution")
sns.countplot(data=df_clean, y="Crop", order=df_clean["Crop"].value_counts().index, ax=axes[1])
axes[1].set_title("Crop Distribution")
plt.tight_layout()
plt.show()

## Cell 16 — EDA observations
After running the charts, write 3–5 observations here. Mention the most common fertility class and any visible differences in soil measurements. Do not claim causation from plots.

## Cell 17 — Correlation matrix

In [ ]:
corr = df_clean[soil_numeric_cols + ["Fertility_Score"]].corr(numeric_only=True)
display(corr.round(2))

## Cell 18 — Correlation heatmap

In [ ]:
plt.figure(figsize=(12,9))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Cell 19 — Feature selection
Target is `Fertility_Class`. Exclude `Sample_ID` (identifier), `Source_Calibration` (provenance text), and `Fertility_Score` (target leakage: it is used to derive the class). Keep measured soil properties and categorical context.

In [ ]:
target = "Fertility_Class"
excluded = ["Sample_ID", "Source_Calibration", "Fertility_Score", target]
X = df_clean.drop(columns=excluded)
y = df_clean[target]
numeric_features = soil_numeric_cols
categorical_features = ["State", "District", "Soil_Type", "Crop"]
print("Input features:", X.columns.tolist())
print("Target:", target)

## Cell 20 — Train-test split (80:20)
Use stratification to preserve the class proportions in both sets. A fixed random state makes the experiment reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).round(3))

## Cell 21 — Preprocessing
Numerical values: median imputation and standardization. Categorical values: most-frequent imputation and one-hot encoding. The pipeline fits preprocessing on training data only.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## Cell 22 — Decision Tree Classifier

## Cell 23 — Random Forest Classifier

## Cell 24 — K-Nearest Neighbors (KNN)

## Cell 25 — Support Vector Machine (SVM)
All models use the same preprocessing and same train/test split.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", class_weight="balanced")
}
trained_models = {}
rows = []
for name, classifier in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    trained_models[name] = pipe
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_test, pred, average="macro", zero_division=0)
    })
results_df = pd.DataFrame(rows).sort_values("F1_macro", ascending=False)
display(results_df.round(4))

## Cell 26 — Model performance comparison chart

In [ ]:
plot_df = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(11,5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0,1)
plt.title("Comparison of Four Classification Models")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Cell 27 — Select the best model
Choose the model with the highest macro F1-score. Macro F1 treats each class equally, which matters because the three classes are imbalanced.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_pred = best_model.predict(X_test)
print("Selected model:", best_model_name)
print("Reason: highest macro F1-score among the four tested models.")

## Cell 28 — Classification report

In [ ]:
print(classification_report(y_test, best_pred, zero_division=0))

## Cell 29 — Confusion matrix

In [ ]:
labels = ["Low Fertility", "Medium Fertility", "High Fertility"]
cm = confusion_matrix(y_test, best_pred, labels=labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — " + best_model_name)
plt.tight_layout()
plt.show()

## Cell 30 — Feature contribution (permutation importance)
Permutation importance estimates how much the test macro-F1 score drops when one input feature is shuffled. It is predictive importance, not proof of causation.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test,
    n_repeats=10, random_state=42, scoring="f1_macro")
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)
display(importance_df)
plt.figure(figsize=(9,6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.show()

## Cell 31 — Prediction for an unseen soil sample
The first test row is used only to demonstrate the prediction workflow. Replace it with a genuinely new sample when you have laboratory measurements.

In [ ]:
new_sample = X_test.iloc[[0]].copy()
display(new_sample)
predicted_class = best_model.predict(new_sample)[0]
print("Predicted fertility class:", predicted_class)

## Cell 32 — Interactive prediction UI
Run this cell in Jupyter. It creates input fields for soil properties and dropdowns for categorical values.

In [ ]:
# If widgets are unavailable, run: %pip install ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

cat_widgets = {
    "State": widgets.Dropdown(options=sorted(df_clean["State"].unique()), description="State"),
    "District": widgets.Dropdown(options=sorted(df_clean["District"].unique()), description="District"),
    "Soil_Type": widgets.Dropdown(options=sorted(df_clean["Soil_Type"].unique()), description="Soil"),
    "Crop": widgets.Dropdown(options=sorted(df_clean["Crop"].unique()), description="Crop")
}
num_widgets = {
    col: widgets.FloatText(value=float(df_clean[col].median()), description=col[:18],
                           style={"description_width":"initial"})
    for col in soil_numeric_cols
}
button = widgets.Button(description="Predict Fertility", button_style="success")
out = widgets.Output()

def run_prediction(_):
    with out:
        clear_output()
        row = {k: w.value for k, w in cat_widgets.items()}
        row.update({k: w.value for k, w in num_widgets.items()})
        sample = pd.DataFrame([row], columns=X.columns)
        print("Predicted class:", best_model.predict(sample)[0])

button.on_click(run_prediction)
display(widgets.VBox([*cat_widgets.values(), *num_widgets.values(), button, out]))

## Cell 33 — Save model and result files

In [ ]:
joblib.dump(best_model, "soil_fertility_best_model.joblib")
results_df.to_csv("model_comparison_results.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)
print("Saved model and result CSV files.")

## Cell 34 — Final conclusion
After running the notebook, report the selected model and its actual test metrics from the outputs. Explain which model performed best and why macro F1-score was considered.

## Cell 35 — Limitations and future scope
The supplied data is source-calibrated synthetic teaching data, not 1,000 authentic laboratory samples. The target label is derived from N, P, K, organic carbon, pH and EC. Therefore, model performance demonstrates learning of the supplied label pattern, not proven real-world soil diagnosis. Future work should validate on authentic laboratory samples and retain source provenance.